# Dedekind DSL: Formal Tier

> *Write it like Python. Reason about it like math. Realize it when you mean it.*

Mathematical set-builder notation with explicit intensional → extensional realization.
Companion to [issue #241](https://github.com/vincentk/dedekind/issues/241).

In [ ]:
import sys
from pathlib import Path

try:
    from dedekind import Ensemble
except ModuleNotFoundError:
    candidate_roots = [Path.cwd(), *Path.cwd().parents]
    for root in candidate_roots:
        candidate = root / "python"
        if (candidate / "dedekind").exists():
            sys.path.insert(0, str(candidate))
            break
    from dedekind import Ensemble

print("Imported formal-tier primitive: Ensemble")
print("Operators: | (union)  & (intersection)  - (difference)  <= (subset)")
print("Methods:   .comprehension()  .restrict()  .map_to()  .realize()  .to_dataframe()")

## Part 1: Extensional Ensembles and Operator Syntax

Finite sets defined by explicit enumeration. Python operator syntax (`|`, `&`, `-`) provides
the easy surface; named methods (`.union()`, `.intersection()`, `.difference()`) are the
formal aliases (issue #241 operator/method parity).

In [ ]:
A = Ensemble.from_members(1, 2, 3, 4, 5)
B = Ensemble.from_members(3, 4, 5, 6, 7)

U = A | B   # union:        A ∪ B
I = A & B   # intersection: A ∩ B
D = A - B   # difference:   A \ B

print(f"A        = {A}")
print(f"B        = {B}")
print(f"A ∪ B    = {U.realize()}")
print(f"A ∩ B    = {I.realize()}")
print(f"A \\ B   = {D.realize()}")
print(f"A ⊆ A∪B? {A <= U}")
print(f"B ⊆ A∩B? {B <= I}")

## Part 2: Intensional Definitions via Comprehension

Set-builder notation: `Ensemble.comprehension(universe, predicate)` mirrors the mathematical
$\{ x \in U \mid P(x) \}$ form. The definition stays intentional until `.realize()` is called —
the explicit realization boundary.

In [ ]:
# {n ∈ [1..30] | n ≡ 0 (mod 2)}  —  evens
E = Ensemble.comprehension(range(1, 31), lambda n: n % 2 == 0)

# {n ∈ [1..30] | n ≡ 1 (mod 2)}  —  odds
O = Ensemble.comprehension(range(1, 31), lambda n: n % 2 == 1)

# Comprehensions partition the universe: E ∪ O = {1..30}, E ∩ O = ∅
EuO = (E | O).realize()
EiO = (E & O).realize()

print(f"Evens  E = {E.realize()}")
print(f"Odds   O = {O.realize()}")
print(f"E ∪ O    = {EuO}  (len={len(EuO)})")
print(f"E ∩ O    = {EiO}  (∅ ↔ empty list: {EiO == []})")

## Part 3: Morphism Image

`.map_to(f)` applies a morphism $f : S \to T$ to produce the image set $f(S) = \{ f(x) \mid x \in S \}$.
Chaining `.restrict().map_to()` corresponds to a restricted morphism.

In [ ]:
# Image under squaring morphism: Q = {n² | n ∈ E}
Q = E.map_to(lambda n: n * n)

# Restrict then map: squares of evens ≤ 10
Q_small = E.restrict(lambda n: n <= 10).map_to(lambda n: n * n)

print(f"Squares of evens Q        = {Q.realize()}")
print(f"Squares of evens ≤ 10     = {Q_small.realize()}")

# DataFrame rendering
print()
print("Evens as DataFrame:")
print(E.to_dataframe("n"))

In [ ]:
# --- Assertions ---
assert (A | B).realize() == [1, 2, 3, 4, 5, 6, 7]
assert (A & B).realize() == [3, 4, 5]
assert (A - B).realize() == [1, 2]

evens = E.realize()
odds  = O.realize()
assert len(evens) == 15
assert len(odds)  == 15
assert sorted(evens + odds) == list(range(1, 31))

squares = Q.realize()
assert len(squares) == 15
assert squares[0] == 4    # 2² = 4
assert squares[-1] == 900 # 30² = 900

assert A <= (A | B)
assert not (B <= (A & B))

print("formal-tier invariants verified")
print("notebook-04-ok")

## Part 4: Minimal DSL v1 (Agent-Oriented)

> This section ideates a minimal operational DSL for agent scripting. The goal is not
to replace Python, but to define a small, auditable plan/apply/verify surface.

> See spec: `docs/python/minimal-dsl-v1-spec.md`

Core lifecycle:
1. `plan` - compute ordered actions (no side effects)
2. `apply` - execute idempotent writes
3. `verify` - assert postconditions/invariants

Core edge kinds:
- `correlates_with`
- `likely_enables`
- `blocks`
- `depends_on`
- `causal_hypothesis`
- `granger_candidate`

In [1]:
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class Action:
    kind: str
    target: int
    payload: Dict[str, Any]
    reason: str
    idempotency_key: str

def plan_quadrant_labeling(assignments: Dict[int, str]) -> List[Action]:
    actions: List[Action] = []
    for issue, label in sorted(assignments.items()):
        actions.append(
            Action(
                kind="tag",
                target=issue,
                payload={"label": label},
                reason="apply correlation x causality matrix",
                idempotency_key=f"tag:{issue}:{label}",
            )
        )
    return actions

def verify_one_quadrant(labels_by_issue: Dict[int, List[str]]) -> bool:
    quadrants = {
        "priority:q1-high-corr-high-causal",
        "priority:q2-high-corr-low-causal",
        "priority:q3-low-corr-high-causal",
        "priority:q4-low-corr-low-causal",
    }
    for issue, labels in labels_by_issue.items():
        count = sum(1 for label in labels if label in quadrants)
        if count != 1:
            print(f"issue #{issue} violates one-quadrant invariant (count={count})")
            return False
    return True

# ideated assignments from current backlog posture
assignments = {
    129: "priority:q1-high-corr-high-causal",
    161: "priority:q1-high-corr-high-causal",
    171: "priority:q2-high-corr-low-causal",
    300: "priority:q3-low-corr-high-causal",
    230: "priority:q4-low-corr-low-causal",
}

actions = plan_quadrant_labeling(assignments)
print("planned actions:")
for action in actions:
    print(action)

labels_by_issue = {
    129: ["priority:q1-high-corr-high-causal", "math"],
    161: ["priority:q1-high-corr-high-causal"],
    171: ["priority:q2-high-corr-low-causal"],
    300: ["priority:q3-low-corr-high-causal", "linear-algebra"],
    230: ["priority:q4-low-corr-low-causal"],
}

print("one-quadrant invariant:", verify_one_quadrant(labels_by_issue))

planned actions:
Action(kind='tag', target=129, payload={'label': 'priority:q1-high-corr-high-causal'}, reason='apply correlation x causality matrix', idempotency_key='tag:129:priority:q1-high-corr-high-causal')
Action(kind='tag', target=161, payload={'label': 'priority:q1-high-corr-high-causal'}, reason='apply correlation x causality matrix', idempotency_key='tag:161:priority:q1-high-corr-high-causal')
Action(kind='tag', target=171, payload={'label': 'priority:q2-high-corr-low-causal'}, reason='apply correlation x causality matrix', idempotency_key='tag:171:priority:q2-high-corr-low-causal')
Action(kind='tag', target=230, payload={'label': 'priority:q4-low-corr-low-causal'}, reason='apply correlation x causality matrix', idempotency_key='tag:230:priority:q4-low-corr-low-causal')
Action(kind='tag', target=300, payload={'label': 'priority:q3-low-corr-high-causal'}, reason='apply correlation x causality matrix', idempotency_key='tag:300:priority:q3-low-corr-high-causal')
one-quadrant inv